In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
sixmetrics_from_sub_variables_signlessCorr_composite5.py
--------------------------------------------------------
変数方向（features）の six-metrics + composite5 を
「距離空間（signlessCorr: d = 1 - |r|）」ベースで評価するスクリプト。

- クラスタ割当:
    STEP3.2_signlessCorr の cluster_labels_matrix_variables_{OH,FP}_<runid>.csv

- 距離空間:
    preprocessed_features_{OH,FP}.csv から
    変数間相関 r を計算 → d = 1 - |r|。
    各変数 j について、「距離プロファイル」
        sim_jk = max(d) - d_jk （自己は除外 → NaN → 0）
    を正規化して確率分布ベクトル p_j とし、
    six-metrics の cosine / JS divergence をこの分布ベクトル間で計算。

- 出力:
    <BASE>/plots_sixmetrics_sub_variables_signlessCorr/
        Table_sixmetrics_variables_signlessCorr_bidirectional_composite5_sub.csv
        Fig_variables_signlessCorr_<direction>_<metric>.png/pdf
"""

from pathlib import Path
import argparse
import itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------- 共通設定（サンプル版と同じ） -------------
DEFAULT_BASE = Path(
    "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/"
    "cal_cluster_No/20251026_under_25clusters_program_and_result/"
    "for_checking_20251127/sub"
)
DEFAULT_FEAT_ROOT = DEFAULT_BASE.parent / "data" / "for_MDS_STEP2"
DEFAULT_RUN_ID = "20251130_153500"

TARGETS = {"PCEmax", "Jsc", "Voc", "FF"}

METRICS_ORDER = [
    "mean_purity",
    "pair_major_same_rate",
    "pair_mean_cosine_similarity",
    "cluster_median_cohesive_ratio",
    "mean_entropy",
    "pair_mean_JS_divergence",
    "composite5",
]

IDX_ORDER = ["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"]


def debug(msg: str):
    print(f"[DEBUG] {msg}")


def cosine(u: np.ndarray, v: np.ndarray) -> float:
    nu = np.linalg.norm(u)
    nv = np.linalg.norm(v)
    if nu == 0 or nv == 0:
        return 0.0
    return float(np.dot(u, v) / (nu * nv))


def js_divergence(p: np.ndarray, q: np.ndarray) -> float:
    m = 0.5 * (p + q)

    def kl(a, b):
        nz = (a > 0) & (b > 0)
        return np.sum(a[nz] * np.log2(a[nz] / b[nz]))

    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


def entropy_from_counts(counts: pd.Series) -> float:
    tot = counts.sum()
    if tot <= 0:
        return 0.0
    p = counts[counts > 0] / tot
    return float(-np.sum(p * np.log2(p)))


def distance_to_probability_profiles(D: pd.DataFrame) -> pd.DataFrame:
    """
    signlessCorr 距離行列 D (index = id, columns = id) から
    各行 i の距離プロファイルを確率分布に変換。
    """
    D = D.copy()
    ids = D.index.astype(str)
    D.index = ids
    D.columns = ids

    n = len(D)
    D.values[np.arange(n), np.arange(n)] = np.nan

    maxD = np.nanmax(D.values)
    if not np.isfinite(maxD) or maxD <= 0:
        S = pd.DataFrame(
            np.ones((n, n), dtype=float) / n,
            index=ids,
            columns=ids,
        )
        return S

    S = maxD - D
    S = S.fillna(0.0)

    row_sum = S.sum(axis=1)
    row_sum = row_sum.replace(0, np.nan)
    P = S.div(row_sum, axis=0).fillna(0.0)
    return P


def six_metrics_direction(
    left_assign: pd.DataFrame,
    right_assign: pd.DataFrame,
    right_vec_prob: pd.DataFrame,
    mode: str,
    index: str,
    thr=(0.6, 0.6, 0.5),
):
    L = left_assign[
        (left_assign["mode"] == mode) & (left_assign["index"] == index)
    ][["id", "cluster"]].rename(columns={"cluster": "L"})
    R = right_assign[
        (right_assign["mode"] == mode) & (right_assign["index"] == index)
    ][["id", "cluster"]].rename(columns={"cluster": "R"})

    M = pd.merge(L, R, on="id", how="inner")
    if M.empty:
        return None

    pur_list, ent_list = [], []
    cos_vals, js_vals, coh_ratio = [], [], []
    pair_same_n = 0
    pair_tot_n = 0

    for l, grp in M.groupby("L"):
        vc = grp["R"].value_counts()
        purity = vc.max() / vc.sum() if vc.sum() > 0 else np.nan
        ent = entropy_from_counts(vc)
        pur_list.append(purity)
        ent_list.append(ent)

        ids = grp["id"].tolist()
        if len(ids) >= 2:
            sub = right_vec_prob.loc[right_vec_prob.index.intersection(ids)]
            mat = sub.to_numpy()
            same = 0
            total = 0
            coh = 0
            for i, j in itertools.combinations(range(mat.shape[0]), 2):
                total += 1
                same += int(grp["R"].iloc[i] == grp["R"].iloc[j])
                c = cosine(mat[i], mat[j])
                js = js_divergence(mat[i], mat[j])
                cos_vals.append(c)
                js_vals.append(js)
                cond1 = (grp["R"].iloc[i] == grp["R"].iloc[j]) and (purity >= thr[0])
                cond2 = c >= thr[1]
                cond3 = js <= thr[2]
                if cond1 or cond2 or cond3:
                    coh += 1
            pair_same_n += same
            pair_tot_n += total
            if total > 0:
                coh_ratio.append(coh / total)

    return {
        "mean_purity": np.nanmean(pur_list) if pur_list else np.nan,
        "pair_major_same_rate": (pair_same_n / pair_tot_n) if pair_tot_n > 0 else np.nan,
        "pair_mean_cosine_similarity": np.nanmean(cos_vals) if cos_vals else np.nan,
        "cluster_median_cohesive_ratio": np.nanmedian(coh_ratio)
        if coh_ratio
        else np.nan,
        "mean_entropy": np.nanmean(ent_list) if ent_list else np.nan,
        "pair_mean_JS_divergence": np.nanmean(js_vals) if js_vals else np.nan,
        "k_left": M["L"].nunique(),
        "k_right": M["R"].nunique(),
        "n": len(M),
    }


# ------------- cluster_labels_matrix_variables_* → tidy -------------
def read_variable_assignments_from_cluster_matrix(
    base: Path, run_id: str, dataset: str
) -> pd.DataFrame:
    """
    base/03_clustering_STEP3.2_signlessCorr/run_<run_id>/variables/<dataset>/
      cluster_labels_matrix_variables_<dataset>_<run_id>.csv

    wide → tidy (id, cluster, mode, index)
    mode = "linear_top3" / "linear_cumeig" / "nonlinear_top3" / "nonlinear_cumeig" など
    index = "silhouette" など
    """
    run_dir = (
        base
        / "03_clustering_STEP3.2_signlessCorr"
        / f"run_{run_id}"
        / "variables"
        / dataset
    )
    fname = f"cluster_labels_matrix_variables_{dataset}_{run_id}.csv"
    path = run_dir / fname

    if not path.exists():
        debug(f"Cluster matrix file missing for variables {dataset}: {path}")
        return pd.DataFrame(columns=["id", "cluster", "mode", "index"])

    debug(f"Reading variable cluster matrix: {path}")
    df = pd.read_csv(path)

    if "id" not in df.columns:
        df.rename(columns={df.columns[0]: "id"}, inplace=True)

    df["id"] = df["id"].astype(str)

    rows = []
    for col in df.columns:
        if col == "id":
            continue
        parts = col.split("_")
        if len(parts) < 2:
            continue
        mode = "_".join(parts[:-1])
        idx = parts[-1]
        tmp = pd.DataFrame(
            {
                "id": df["id"],
                "cluster": pd.to_numeric(df[col], errors="coerce").astype("Int64"),
                "mode": mode,
                "index": idx,
            }
        )
        rows.append(tmp)

    if not rows:
        debug(f"No cluster columns found in {path}")
        return pd.DataFrame(columns=["id", "cluster", "mode", "index"])

    A = pd.concat(rows, ignore_index=True)
    A = A.dropna(subset=["cluster"])
    A["cluster"] = A["cluster"].astype(int)
    A["id"] = A["id"].astype(str)
    return A


# ------------- 特徴量 → 変数間 signlessCorr 距離 -------------
def load_feature_matrix(feat_root: Path, kind: str) -> pd.DataFrame:
    """
    kind: 'OH' or 'FP'
    preprocessed_features_{kind}.csv を読み込み、
    目的変数(PCEmax,Jsc,Voc,FF)を除いた変数を対象とする。
    rows = samples, cols = variables
    """
    path = feat_root / f"preprocessed_features_{kind}.csv"
    if not path.exists():
        debug(f"Feature file missing: {path}")
        return pd.DataFrame()

    df = pd.read_csv(path, header=0, index_col=0)
    keep = [c for c in df.columns if c not in TARGETS]
    df = df.loc[:, keep]
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def compute_signlesscorr_variable_distance(df: pd.DataFrame) -> pd.DataFrame:
    """
    変数間の signlessCorr 距離 d = 1 - |r|。
    df: rows = samples, cols = variables
    """
    if df.empty:
        return pd.DataFrame()
    R = df.corr()  # 列間相関 → 変数間
    R = R.fillna(0.0)
    D = 1.0 - np.abs(R)
    D = D.clip(lower=0.0, upper=2.0)
    return D


# ------------- 図化 -------------
def plot_bar_by_index(MET: pd.DataFrame, metric: str, title_prefix: str, outbase: Path):
    sub = MET.copy()
    if metric not in sub.columns:
        return

    directions = ["OH→FP", "FP→OH"]
    x = np.arange(len(IDX_ORDER), dtype=float)
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 4.5), dpi=300)

    vals_L = []
    vals_R = []
    for ix in IDX_ORDER:
        rL = sub[(sub["direction"] == directions[0]) & (sub["index"] == ix)]
        rR = sub[(sub["direction"] == directions[1]) & (sub["index"] == ix)]

        def get_val(row):
            if row.empty:
                return np.nan
            v = row.iloc[0][metric]
            return float(v) if pd.notna(v) else np.nan

        vals_L.append(get_val(rL))
        vals_R.append(get_val(rR))

    ax.bar(x - width / 2, vals_L, width=width, label=directions[0])
    ax.bar(x + width / 2, vals_R, width=width, label=directions[1])

    ax.set_xticks(x)
    ax.set_xticklabels(IDX_ORDER, rotation=0)
    ax.set_ylabel(metric)
    ax.set_title(f"{title_prefix}: {metric}")
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5, axis="y")
    ax.legend(loc="best")

    finite_vals = [v for v in (vals_L + vals_R) if np.isfinite(v)]
    if finite_vals:
        vmax = max(finite_vals)
    else:
        vmax = 1.0
    if metric == "mean_entropy" or metric == "pair_mean_JS_divergence":
        ytop = float(min(max(vmax * 1.2, 0.2), 3.0))
    else:
        ytop = float(min(max(vmax * 1.1, 0.4), 1.05))
    ax.set_ylim(0.0, ytop)

    fig.tight_layout()
    fig.savefig(outbase.with_suffix(".png"))
    fig.savefig(outbase.with_suffix(".pdf"))
    plt.close(fig)


# ------------- メイン -------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument(
        "--base",
        type=str,
        default=str(DEFAULT_BASE),
        help="for_checking_20251127/sub のパス",
    )
    ap.add_argument(
        "--featroot",
        type=str,
        default=str(DEFAULT_FEAT_ROOT),
        help="preprocessed_features_*.csv のあるルート (for_MDS_STEP2)",
    )
    ap.add_argument(
        "--runid",
        type=str,
        default=DEFAULT_RUN_ID,
        help="STEP3.2_signlessCorr の run ID (例: 20251130_153500)",
    )
    args, _ = ap.parse_known_args()

    BASE = Path(args.base).resolve()
    FEAT_ROOT = Path(args.featroot).resolve()
    RUN_ID = args.runid

    out_dir = BASE / "plots_sixmetrics_sub_variables_signlessCorr"
    out_dir.mkdir(parents=True, exist_ok=True)
    print("[INFO] Output dir:", out_dir)

    # 1) 変数クラスタ割当
    print("[INFO] Reading variable-wise assignments from cluster_labels_matrix (OH/FP)...")
    A_OH = read_variable_assignments_from_cluster_matrix(BASE, RUN_ID, "OH")
    A_FP = read_variable_assignments_from_cluster_matrix(BASE, RUN_ID, "FP")
    print("[INFO] OH variable assignments:", A_OH.shape, "FP variable assignments:", A_FP.shape)

    if A_OH.empty or A_FP.empty:
        print("❌ 変数方向の割当が読み込めませんでした。パスやファイル名を確認してください。")
        return

    modes = sorted(set(A_OH["mode"]) & set(A_FP["mode"]))
    idxs = sorted(set(A_OH["index"]) & set(A_FP["index"]))
    print("[INFO] modes:", modes)
    print("[INFO] indices:", idxs)

    # 2) 特徴量読み込み → 変数間 signlessCorr 距離 → プロファイル
    X_OH = load_feature_matrix(FEAT_ROOT, "OH")
    X_FP = load_feature_matrix(FEAT_ROOT, "FP")
    print("[INFO] Features (samples x variables): OH", X_OH.shape, "FP", X_FP.shape)

    if X_OH.empty or X_FP.empty:
        print("❌ 特徴量が読み込めませんでした。--featroot を確認してください。")
        return

    # 列名を文字列
    X_OH.columns = X_OH.columns.astype(str)
    X_FP.columns = X_FP.columns.astype(str)

    D_OH = compute_signlesscorr_variable_distance(X_OH)
    D_FP = compute_signlesscorr_variable_distance(X_FP)

    if D_OH.empty or D_FP.empty:
        print("❌ signlessCorr 距離の計算に失敗しました。")
        return

    P_OH = distance_to_probability_profiles(D_OH)
    P_FP = distance_to_probability_profiles(D_FP)

    # 共通変数 ID のみにそろえる（右側ベクトル用）
    common_vars_OH = sorted(set(A_OH["id"]) & set(P_OH.index))
    common_vars_FP = sorted(set(A_FP["id"]) & set(P_FP.index))
    if not common_vars_OH or not common_vars_FP:
        print("❌ 共通の変数名が不足しています。")
        return

    P_OH = P_OH.loc[common_vars_OH, :]
    P_FP = P_FP.loc[common_vars_FP, :]

    # 3) six-metrics（両方向）
    rows = []
    for md in modes:
        for ix in idxs:
            a = six_metrics_direction(A_OH, A_FP, P_FP, md, ix)  # OH→FP
            b = six_metrics_direction(A_FP, A_OH, P_OH, md, ix)  # FP→OH
            if a:
                rows.append({"direction": "OH→FP", "mode": md, "index": ix, **a})
            if b:
                rows.append({"direction": "FP→OH", "mode": md, "index": ix, **b})

    if not rows:
        print("❌ six-metrics が計算できませんでした。")
        return

    MET = pd.DataFrame(rows)

    # 4) entropy_coh, js_coh, composite5
    with np.errstate(divide="ignore", invalid="ignore"):
        kr = MET["k_right"].clip(lower=2)
        maxH = np.log2(kr)
        MET["entropy_coh"] = 1.0 - (MET["mean_entropy"] / maxH)
    MET["entropy_coh"] = MET["entropy_coh"].clip(0, 1)

    MET["js_coh"] = 1.0 - MET["pair_mean_JS_divergence"]
    MET["js_coh"] = MET["js_coh"].clip(0, 1)

    MET["composite5"] = MET[
        [
            "mean_purity",
            "pair_major_same_rate",
            "pair_mean_cosine_similarity",
            "cluster_median_cohesive_ratio",
            "entropy_coh",
        ]
    ].mean(axis=1)

    out_csv = out_dir / "Table_sixmetrics_variables_signlessCorr_bidirectional_composite5_sub.csv"
    MET.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("✅ Saved table:", out_csv)

    # 5) 図化
    for m in METRICS_ORDER:
        outbase = out_dir / f"Fig_variables_signlessCorr_{m}"
        plot_bar_by_index(MET, m, "Variables (signlessCorr)", outbase)

    print("✅ Saved figures (PNG/PDF) to:", out_dir)


if __name__ == "__main__":
    main()

[INFO] Output dir: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/plots_sixmetrics_sub_variables_signlessCorr
[INFO] Reading variable-wise assignments from cluster_labels_matrix (OH/FP)...
[DEBUG] Reading variable cluster matrix: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/03_clustering_STEP3.2_signlessCorr/run_20251130_153500/variables/OH/cluster_labels_matrix_variables_OH_20251130_153500.csv
[DEBUG] Reading variable cluster matrix: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/03_clustering_STEP3.2_signlessCorr/run_20251130_153500/variables/FP/cluster_labels_matrix_variables_FP_20251130_153500.csv
[INFO] OH variable assignments: (9456, 4) FP variable assignments: (6024, 4)
[INFO] modes: ['linear_cumeig', 'linear_top3', 'nonlinear_cu